# Simulations, Models, and Pipeline Configuration

This notebook covers:
- Browsing built-in simulation, model, and dust configurations
- Choosing and combining simulations and models
- Controlling pipeline stages with `RunFlags`
- Customising output paths and SLURM settings

In [ ]:
from galform_execution.submit_galform_job import (
    GalformSubmitter,
    RunFlags,
    SIMULATION_CONFIGS,
    MODEL_CONFIGS,
    DUST_CONFIGS,
    load_run_flags_config,
    _default_galform_dir,
)

GALFORM_DIR = _default_galform_dir()

## 1. Available simulations

Each simulation entry defines tree paths, cosmological parameters, default snapshot list (`iz_list`), and subvolume range (`nvol_range`). These are resolved automatically when you name a simulation.

In [ ]:
print(f"{'Simulation':<25} {'iz_list':<50} {'nvol_range'}")
print("-" * 90)
for name, cfg in sorted(SIMULATION_CONFIGS.items()):
    iz_str = str(cfg.iz_list) if cfg.iz_list else "(not set)"
    if len(iz_str) > 47:
        iz_str = iz_str[:44] + "..."
    print(f"{name:<25} {iz_str:<50} {cfg.nvol_range}")

## 2. Available models

Each model maps to a base `.input.ref` file, a dust profile, and optional parameter overrides (`extra_replacements`) that are injected automatically into every run.

In [ ]:
print(f"{'Model':<30} {'Base input file':<48} {'Dust profile':<12} {'Extra replacements'}")
print("-" * 110)
for name, cfg in sorted(MODEL_CONFIGS.items()):
    dust_key = next(
        (k for k, d in DUST_CONFIGS.items() if d == cfg.dust_params), "custom"
    )
    extras = str(cfg.extra_replacements) if cfg.extra_replacements else "-"
    print(f"{name:<30} {cfg.base_inputs_file:<48} {dust_key:<12} {extras}")

## 3. Available dust profiles

In [ ]:
print(f"{'Profile':<12} {'fcloud':<10} {'rfacburst':<12} {'tesc_disk':<12} {'tesc_burst':<12} {'dustfile'}")
print("-" * 90)
for name, d in DUST_CONFIGS.items():
    print(f"{name:<12} {d.fcloud:<10} {d.rfacburst:<12} {d.tesc_disk:<12} {d.tesc_burst:<12} {d.dustfile}")

## 4. Choosing a simulation and model

When `iz_list` and `nvol` are omitted, the submitter uses the simulation's built-in defaults.

In [ ]:
# P-Millennium with updated merger histories
s = GalformSubmitter(GALFORM_DIR, nbody_sim="L800", model="lc16.newmg",
                     output_folder_name="L800_Production")
print(f"iz_list    : {s.iz_list}")
print(f"nvol_range : {s.nvol_range}  ({s.nvol_count} tasks per snapshot)")

In [ ]:
# Override the default snapshot list to submit only specific snapshots
s_partial = GalformSubmitter(GALFORM_DIR, nbody_sim="L800", model="lc16.newmg",
                              iz_list=[271, 207, 155],
                              output_folder_name="L800_HighZ")
print(f"iz_list : {s_partial.iz_list}")

In [ ]:
# FLAMINGO on COSMA8
s_flamingo = GalformSubmitter(GALFORM_DIR, nbody_sim="FLAMINGO-L1000N1800", model="lc16",
                               output_base_dir="/cosma8/data/dp004/dc-hick2",
                               partition="cosma8", account="dp004",
                               output_folder_name="FLAMINGO_Run")
print(f"iz_list : {s_flamingo.iz_list}")
print(f"Output  : {s_flamingo.models_dir}")

## 5. Controlling pipeline stages with RunFlags

The pipeline runs several executables after the main GALFORM code. `RunFlags` controls which ones are active. Defaults are loaded from `config/run_flags.json`.

| Flag | Executable | Default |
|---|---|---|
| `galform` | `galform2` | `True` |
| `neta` | `neta_ave_disk/burst` (dust extinction) | `True` |
| `lum_fun` | `sample_gals` (luminosity functions) | `True` |
| `study_stellar_mass_function` | `sample_gals` (SMF) | `True` |
| `dust_props` | dust properties output | `False` |
| `samp_z0` | z=0 galaxy catalogue | `False` |
| `cosmicsed` | cosmic SED | `False` |

In [ ]:
print(load_run_flags_config())

In [ ]:
# Run GALFORM + neta only — useful for debugging or when post-processing runs separately
s_debug = GalformSubmitter(
    GALFORM_DIR, nbody_sim="Mill2", model="lc16", iz=40, nvol="1-4",
    run_flags=RunFlags(galform=True, neta=True, lum_fun=False,
                       study_stellar_mass_function=False),
    output_folder_name="Mill2_Debug",
)
s_debug.submit_all_jobs(dry_run=True)

In [ ]:
# Full pipeline including dust properties and z=0 catalogue
s_full = GalformSubmitter(
    GALFORM_DIR, nbody_sim="Mill2", model="lc16", iz=40, nvol="1-64",
    run_flags=RunFlags(galform=True, neta=True, lum_fun=True,
                       study_stellar_mass_function=True,
                       dust_props=True, samp_z0=True),
    output_folder_name="Mill2_Full",
)
s_full.submit_all_jobs(dry_run=True)

## 6. SLURM and output path settings

All defaults target COSMA5 (`/cosma5/data/durham/$USER`). Override for COSMA7/8 or custom allocations.

In [ ]:
s_c7 = GalformSubmitter(
    GALFORM_DIR,
    nbody_sim="L800", model="lc16.newmg",
    output_base_dir="/cosma7/data/dp004/dc-hick2",
    log_path="/cosma7/data/dp004/dc-hick2/logs/Production_v2",
    output_folder_name="Production_v2",
    partition="cosma7", account="dp004", walltime="48:00:00",
)
print(f"Output : {s_c7.models_dir}")
print(f"Logs   : {s_c7.log_path}")
print(f"Queue  : {s_c7.partition} / {s_c7.account}  wall={s_c7.walltime}")

## 7. Retry configuration

SLURM can transiently reject submissions under scheduler pressure. The submitter retries with exponential backoff automatically. Defaults: 4 retries, 15 s base delay, ×2 backoff.

In [ ]:
s_retry = GalformSubmitter(
    GALFORM_DIR,
    nbody_sim="L800", model="lc16.newmg",
    output_folder_name="L800_Production",
    submit_retries=8,
    submit_retry_delay_s=30.0,
    submit_retry_backoff=2.0,
)
print(f"Retries: {s_retry.submit_retries}, base delay: {s_retry.submit_retry_delay_s}s, backoff: x{s_retry.submit_retry_backoff}")

---

**Next:** `03_parameter_studies_and_trees.ipynb` — varying model parameters with `input_overrides`, running parameter grids, multi-output redshifts, and building galaxy merger trees.